In [4]:


import imaplib
import email
from email.header import decode_header

def search_emails(prompt: str, n: int, email_address: str, app_password: str):
    """
    Searches and returns up to 'n' emails based on the given prompt using the IMAP protocol.

    Args:
        prompt (str): The search prompt (e.g., "SUBJECT 'Test'", "FROM 'example@gmail.com'").
        n (int): Number of emails to return.
        email_address (str): Your email address.
        app_password (str): Your app password.

    Returns:
        List[Dict[str, str]]: A list of dictionaries containing email subject, sender, and snippet.
    """
    # Connect to Gmail's IMAP server
    imap = imaplib.IMAP4_SSL("imap.gmail.com")
    
    # Login to your email account
    try:
        imap.login(email_address, app_password)
    except imaplib.IMAP4.error as e:
        raise Exception("Login failed. Check your credentials.") from e

    # Select the inbox (or other folders like 'Sent', 'Spam', etc.)
    imap.select("inbox")

    # Search emails using the prompt
    try:
        status, message_ids = imap.search(None, prompt)
        if status != "OK":
            raise Exception("Failed to search emails.")

        # Convert the message IDs to a list
        message_ids = message_ids[0].split()

        # Fetch up to 'n' emails
        emails = []
        for msg_id in message_ids[-n:]:
            status, msg_data = imap.fetch(msg_id, "(RFC822)")
            if status != "OK":
                continue
            
            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    # Parse the email
                    msg = email.message_from_bytes(response_part[1])
                    subject, encoding = decode_header(msg["Subject"])[0]
                    if isinstance(subject, bytes):
                        subject = subject.decode(encoding or "utf-8")

                    from_ = msg.get("From")
                    snippet = ""

                    # Get email snippet (text/plain part)
                    if msg.is_multipart():
                        for part in msg.walk():
                            content_type = part.get_content_type()
                            content_disposition = str(part.get("Content-Disposition"))
                            if content_type == "text/plain" and "attachment" not in content_disposition:
                                snippet = part.get_payload(decode=True).decode()
                                break
                    else:
                        snippet = msg.get_payload(decode=True).decode()

                    emails.append({
                        "subject": subject,
                        "from": from_,
                        "snippet": snippet[:100]  # First 100 characters of the body
                    })

        # Logout and return emails
        imap.logout()
        return emails

    except Exception as e:
        imap.logout()
        raise e





In [ ]:
# Usage example:

# Gmail credentials
email_address = ""
app_password = ""  # Generated App Password


# Search prompt (e.g., "SUBJECT 'Test'" or "FROM 'example@gmail.com'")
prompt = 'SUBJECT "job application"'

# Number of emails to return
n = 5

try:
    emails = search_emails(prompt, n, email_address, app_password)
    for idx, email_info in enumerate(emails, start=1):
        print(f"Email {idx}:")
        print(f"Subject: {email_info['subject']}")
        print(f"From: {email_info['from']}")
        print(f"Snippet: {email_info['snippet']}\n")
        print("email : ", email_info)
except Exception as e:
    print(f"Error: {e}")